# Student Success Early-Warning Analysis

## Project objective

This project investigates whether machine-learning models can identify students at risk of dropout or unsuccessful academic outcomes using enrolment, early academic and online-learning information.

The analysis is framed as an early-warning decision-support system. Predictions should be used to offer appropriate support to students and should not automatically determine interventions.

## Research questions

1. Can Decision Tree and Random Forest classifiers identify students at risk using information available early enough for intervention?
2. Do student background and early academic attributes or online-engagement behaviour provide stronger early-warning signals?
3. Are the insights from the two datasets complementary or contradictory?

## Reproducibility

All modelling uses `random_state=42`. The two datasets are analysed separately because they come from different institutions and use different structures and outcome definitions.

In [1]:
from pathlib import Path
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

RANDOM_STATE = 42

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

sns.set_theme(style="whitegrid")

print("Python:", sys.version)
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
pandas: 3.0.5
NumPy: 2.5.1
scikit-learn: 1.9.0


In [2]:
# Locate the project root whether the notebook starts in the project
# directory or inside the notebooks directory.
current_directory = Path.cwd().resolve()

if (current_directory / "data" / "raw").exists():
    PROJECT_ROOT = current_directory
elif (current_directory.parent / "data" / "raw").exists():
    PROJECT_ROOT = current_directory.parent
else:
    raise FileNotFoundError(
        "Could not locate data/raw. Open the notebook from the project folder."
    )

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
DROPOUT_PATH = RAW_DATA_DIR / "dropout" / "data.csv"
OULAD_DIR = RAW_DATA_DIR / "oulad"

FILE_PATHS = {
    "dropout": DROPOUT_PATH,
    "assessments": OULAD_DIR / "assessments.csv",
    "courses": OULAD_DIR / "courses.csv",
    "studentAssessment": OULAD_DIR / "studentAssessment.csv",
    "studentInfo": OULAD_DIR / "studentInfo.csv",
    "studentRegistration": OULAD_DIR / "studentRegistration.csv",
    "studentVle": OULAD_DIR / "studentVle.csv",
    "vle": OULAD_DIR / "vle.csv",
}

file_audit = pd.DataFrame(
    [
        {
            "file": name,
            "path": str(path),
            "exists": path.exists(),
            "size_mb": round(path.stat().st_size / (1024**2), 2)
            if path.exists()
            else np.nan,
        }
        for name, path in FILE_PATHS.items()
    ]
)

display(file_audit)

assert file_audit["exists"].all(), "One or more required dataset files are missing."

print("Project root:", PROJECT_ROOT)
print("All required files were found.")

,file,path,exists,size_mb
0,dropout,C:\Users\fatim\OneDrive\Documents\data science...,True,0.51
1,assessments,C:\Users\fatim\OneDrive\Documents\data science...,True,0.01
2,courses,C:\Users\fatim\OneDrive\Documents\data science...,True,0.00
3,studentAssessment,C:\Users\fatim\OneDrive\Documents\data science...,True,5.43
4,studentInfo,C:\Users\fatim\OneDrive\Documents\data science...,True,3.30
5,studentRegistration,C:\Users\fatim\OneDrive\Documents\data science...,True,1.08
6,studentVle,C:\Users\fatim\OneDrive\Documents\data science...,True,432.81
7,vle,C:\Users\fatim\OneDrive\Documents\data science...,True,0.26


Project root: C:\Users\fatim\OneDrive\Documents\data science
All required files were found.


In [3]:
# The downloaded UCI file uses semicolons as delimiters.
dropout_df = pd.read_csv(DROPOUT_PATH, sep=";")

# Remove accidental whitespace from column names without changing raw data.
dropout_df.columns = dropout_df.columns.str.strip()

print("Dataset shape:", dropout_df.shape)
print("Duplicate rows:", dropout_df.duplicated().sum())
print("\nColumns:")
print(dropout_df.columns.tolist())

display(dropout_df.head())

dropout_missing = (
    dropout_df.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

print("\nColumns containing missing values:")
display(dropout_missing[dropout_missing["missing_count"] > 0])

target_distribution = (
    dropout_df["Target"]
    .value_counts(dropna=False)
    .rename_axis("Target")
    .reset_index(name="count")
)

target_distribution["percentage"] = (
    target_distribution["count"] / len(dropout_df) * 100
).round(2)

print("\nTarget-class distribution:")
display(target_distribution)

Dataset shape: (4424, 37)
Duplicate rows: 0

Columns:
['Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance', 'Previous qualification', 'Previous qualification (grade)', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Admission grade', 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment', 'International', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)', 'Unemployment r

,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Admission grade,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,5,9,127.3,1,0,0,1,1,0,20,0,0,0,0,0,0.000000,0,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,3,3,142.5,1,0,0,0,1,0,19,0,0,6,6,6,14.000000,0,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,9,9,124.8,1,0,0,0,1,0,19,0,0,6,0,0,0.000000,0,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,5,3,119.6,1,0,0,1,0,0,20,0,0,6,8,6,13.428571,0,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,9,9,141.5,0,0,0,1,0,0,45,0,0,6,9,5,12.333333,0,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate



Columns containing missing values:


,missing_count



Target-class distribution:


,Target,count,percentage
0,Graduate,2209,49.93
1,Dropout,1421,32.12
2,Enrolled,794,17.95


In [4]:
# A question mark represents missing information in several OULAD files.
oulad_small_paths = {
    name: path
    for name, path in FILE_PATHS.items()
    if name not in {"dropout", "studentVle"}
}

oulad_tables = {
    name: pd.read_csv(path, na_values=["?"])
    for name, path in oulad_small_paths.items()
}

for table in oulad_tables.values():
    table.columns = table.columns.str.strip()

oulad_summary = pd.DataFrame(
    [
        {
            "table": name,
            "rows": table.shape[0],
            "columns": table.shape[1],
            "missing_cells": int(table.isna().sum().sum()),
            "duplicate_rows": int(table.duplicated().sum()),
        }
        for name, table in oulad_tables.items()
    ]
)

display(oulad_summary)

for name, table in oulad_tables.items():
    print(f"\n{name}:")
    print(table.columns.tolist())
    display(table.head(3))

student_info = oulad_tables["studentInfo"]

oulad_outcomes = (
    student_info["final_result"]
    .value_counts(dropna=False)
    .rename_axis("final_result")
    .reset_index(name="count")
)

oulad_outcomes["percentage"] = (
    oulad_outcomes["count"] / len(student_info) * 100
).round(2)

print("\nOriginal OULAD outcome distribution:")
display(oulad_outcomes)

,table,rows,columns,missing_cells,duplicate_rows
0,assessments,206,6,11,0
1,courses,22,3,0,0
2,studentAssessment,173912,5,173,0
3,studentInfo,32593,12,1111,0
4,studentRegistration,32593,5,22566,0
5,vle,6364,6,10486,0



assessments:
['code_module', 'code_presentation', 'id_assessment', 'assessment_type', 'date', 'weight']


,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2013J,1753,TMA,54.0,20.0
2,AAA,2013J,1754,TMA,117.0,20.0



courses:
['code_module', 'code_presentation', 'module_presentation_length']


,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268


...

,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0



vle:
['id_site', 'code_module', 'code_presentation', 'activity_type', 'week_from', 'week_to']


,id_site,code_module,code_presentation,activity_type,week_from,week_to
0,546943,AAA,2013J,resource,NaN,NaN
1,546712,AAA,2013J,oucontent,NaN,NaN
2,546998,AAA,2013J,resource,NaN,NaN



Original OULAD outcome distribution:


,final_result,count,percentage
0,Pass,12361,37.93
1,Withdrawn,10156,31.16
2,Fail,7052,21.64
3,Distinction,3024,9.28


In [5]:
STUDENT_VLE_PATH = FILE_PATHS["studentVle"]

student_vle_sample = pd.read_csv(
    STUDENT_VLE_PATH,
    na_values=["?"],
    nrows=5,
)

student_vle_sample.columns = student_vle_sample.columns.str.strip()

print("studentVle columns:")
print(student_vle_sample.columns.tolist())

display(student_vle_sample)

# Count rows in chunks without keeping the whole table in memory.
student_vle_row_count = 0

for chunk in pd.read_csv(
    STUDENT_VLE_PATH,
    usecols=["id_student"],
    chunksize=1_000_000,
):
    student_vle_row_count += len(chunk)

print(f"studentVle row count: {student_vle_row_count:,}")

studentVle columns:
['code_module', 'code_presentation', 'id_student', 'id_site', 'date', 'sum_click']


,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1


studentVle row count: 10,655,280


## Target Construction and Leakage Prevention

### Dataset 1: Student Dropout and Academic Success

The original three-class target is retained:

- `Dropout`
- `Enrolled`
- `Graduate`

Keeping `Enrolled` as a separate outcome avoids making the unsupported assumption that students who remain enrolled have already succeeded.

The primary early-warning metric will be recall for the `Dropout` class. Macro F1-score and balanced accuracy will also be used because the three classes are unevenly distributed.

All six second-semester variables are excluded from the main analysis. These variables occur too late to support meaningful early intervention. Enrolment, demographic, socioeconomic and first-semester variables remain available.

### Dataset 2: Open University Learning Analytics Dataset

The original outcomes are transformed into a binary target:

- `At Risk`: `Withdrawn` or `Fail`
- `Successful`: `Pass` or `Distinction`

The prediction point is the end of course day 60. Students who had already unregistered on or before day 60 are excluded because their withdrawal was already known at the prediction point. Including them would not represent a genuine early-warning prediction task.

The following leakage-prevention rules are applied:

- Only VLE interactions from course days 0–60 are used.
- Interactions after day 60 are excluded.
- Assessments must be due by day 60.
- Only submissions available by day 60 are used.
- Students who unregistered on or before day 60 are excluded.
- `final_result` is used only to construct the target.
- `date_unregistration` is used only to define eligibility and is not a predictive feature.
- `id_student` is used only for joining tables and is removed before modelling.

Predictions are intended to support human-led intervention and should not automatically determine decisions about students.

In [6]:
#prepare datatset 1 
DROPOUT_TARGET = "Target"

second_semester_columns = [
    column
    for column in dropout_df.columns
    if "2nd sem" in column
]

print("Second-semester columns excluded:")
for column in second_semester_columns:
    print("-", column)

dropout_model_data = dropout_df.drop(
    columns=second_semester_columns
).copy()

X_dropout_raw = dropout_model_data.drop(
    columns=[DROPOUT_TARGET]
).copy()

y_dropout = dropout_model_data[DROPOUT_TARGET].copy()

dropout_leakage_audit = pd.DataFrame(
    {
        "item": [
            "Original rows",
            "Original predictors",
            "Second-semester predictors excluded",
            "Predictors retained",
            "Target classes",
        ],
        "value": [
            len(dropout_df),
            dropout_df.shape[1] - 1,
            len(second_semester_columns),
            X_dropout_raw.shape[1],
            ", ".join(sorted(y_dropout.unique())),
        ],
    }
)

display(dropout_leakage_audit)

dropout_target_audit = (
    y_dropout.value_counts()
    .rename_axis("Target")
    .reset_index(name="count")
)

dropout_target_audit["percentage"] = (
    dropout_target_audit["count"] / len(y_dropout) * 100
).round(2)

display(dropout_target_audit)

assert len(second_semester_columns) == 6
assert DROPOUT_TARGET not in X_dropout_raw.columns
assert not any("2nd sem" in column for column in X_dropout_raw.columns)

print("Dataset 1 leakage checks passed.")

Second-semester columns excluded:
- Curricular units 2nd sem (credited)
- Curricular units 2nd sem (enrolled)
- Curricular units 2nd sem (evaluations)
- Curricular units 2nd sem (approved)
- Curricular units 2nd sem (grade)
- Curricular units 2nd sem (without evaluations)


,item,value
0,Original rows,4424
1,Original predictors,36
2,Second-semester predictors excluded,6
3,Predictors retained,30
4,Target classes,"Dropout, Enrolled, Graduate"


,Target,count,percentage
0,Graduate,2209,49.93
1,Dropout,1421,32.12
2,Enrolled,794,17.95


Dataset 1 leakage checks passed.


In [7]:
#Construct the OULAD day-60 cohort
OULAD_KEYS = [
    "code_module",
    "code_presentation",
    "id_student",
]

EARLY_WINDOW_END = 60

student_info = oulad_tables["studentInfo"].copy()
student_registration = oulad_tables["studentRegistration"].copy()

# Confirm that each student-course record has one row in each table.
student_info_key_duplicates = student_info.duplicated(
    subset=OULAD_KEYS
).sum()

registration_key_duplicates = student_registration.duplicated(
    subset=OULAD_KEYS
).sum()

print("Duplicate studentInfo keys:", student_info_key_duplicates)
print("Duplicate studentRegistration keys:", registration_key_duplicates)

assert student_info_key_duplicates == 0
assert registration_key_duplicates == 0

oulad_base = student_info.merge(
    student_registration[
        OULAD_KEYS
        + [
            "date_registration",
            "date_unregistration",
        ]
    ],
    on=OULAD_KEYS,
    how="left",
    validate="one_to_one",
)

print("Merged OULAD shape:", oulad_base.shape)
display(oulad_base.head())

Duplicate studentInfo keys: 0
Duplicate studentRegistration keys: 0
Merged OULAD shape: (32593, 14)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,-159.0,NaN
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,-53.0,NaN
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,-92.0,12.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,-52.0,NaN
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,-176.0,NaN


In [8]:
#Remove outcomes known by day 60
# A withdrawal on or before day 60 is already known at the
# prediction point and is therefore excluded from the cohort.
known_withdrawal_by_day_60 = (
    oulad_base["date_unregistration"].notna()
    & oulad_base["date_unregistration"].le(EARLY_WINDOW_END)
)

oulad_day60_cohort = oulad_base.loc[
    ~known_withdrawal_by_day_60
].copy()

cohort_audit = pd.DataFrame(
    {
        "cohort_stage": [
            "Original student-course records",
            "Withdrawals already known by day 60",
            "Records eligible for day-60 prediction",
        ],
        "records": [
            len(oulad_base),
            int(known_withdrawal_by_day_60.sum()),
            len(oulad_day60_cohort),
        ],
    }
)

display(cohort_audit)

withdrawal_timing_audit = pd.crosstab(
    oulad_base["final_result"],
    known_withdrawal_by_day_60,
).rename(
    columns={
        False: "eligible_at_day_60",
        True: "outcome_known_by_day_60",
    }
)

display(withdrawal_timing_audit)

assert (
    len(oulad_day60_cohort)
    + known_withdrawal_by_day_60.sum()
    == len(oulad_base)
)

,cohort_stage,records
0,Original student-course records,32593
1,Withdrawals already known by day 60,6232
2,Records eligible for day-60 prediction,26361


date_unregistration,eligible_at_day_60,outcome_known_by_day_60
final_result,,
Distinction,3024,0
Fail,7044,8
Pass,12361,0
Withdrawn,3932,6224


In [9]:
#Construct the binary OULAD target
at_risk_outcomes = {
    "Withdrawn",
    "Fail",
}

successful_outcomes = {
    "Pass",
    "Distinction",
}

all_expected_outcomes = at_risk_outcomes | successful_outcomes
observed_outcomes = set(
    oulad_day60_cohort["final_result"].dropna().unique()
)

assert observed_outcomes == all_expected_outcomes

oulad_day60_cohort["risk_target"] = np.where(
    oulad_day60_cohort["final_result"].isin(at_risk_outcomes),
    "At Risk",
    "Successful",
)

oulad_target_distribution = (
    oulad_day60_cohort["risk_target"]
    .value_counts()
    .rename_axis("risk_target")
    .reset_index(name="count")
)

oulad_target_distribution["percentage"] = (
    oulad_target_distribution["count"]
    / len(oulad_day60_cohort)
    * 100
).round(2)

display(oulad_target_distribution)

target_mapping = pd.DataFrame(
    {
        "original_outcome": [
            "Withdrawn",
            "Fail",
            "Pass",
            "Distinction",
        ],
        "binary_target": [
            "At Risk",
            "At Risk",
            "Successful",
            "Successful",
        ],
    }
)

display(target_mapping)

assert oulad_day60_cohort["risk_target"].isna().sum() == 0
assert set(oulad_day60_cohort["risk_target"].unique()) == {
    "At Risk",
    "Successful",
}

print("Dataset 2 target construction checks passed.")

,risk_target,count,percentage
0,Successful,15385,58.36
1,At Risk,10976,41.64


,original_outcome,binary_target
0,Withdrawn,At Risk
1,Fail,At Risk
2,Pass,Successful
3,Distinction,Successful


Dataset 2 target construction checks passed.
